In [0]:
%pip install openpyxl

# JOH Reconciliation Tests

JOH archives **judicial officers** (judges), not appeal cases. Source is `ariadm_arm_joh.raw_adjudicator` joined with `raw_adjudicatorrole`, NOT `raw_appealcase`. That's why JOH is reconciled separately from the case-based overall reconciliation.

**Equation:** `count(Adjudicator passing segmentation filter) == count(gold_judicial_officer_with_json)`

**Segmentation filter** (from the JOH LLD):
```sql
WHERE jr.Role NOT IN (7, 8) OR jr.Role IS NULL
```

**Expected volume** (LLD estimate): ~1,860 records — informational baseline only.

Each test is in its own cell. Results land in `test_automation_runs2` / `test_automation_results2` / `test_automation_perfresults2`, plus CSV/XLSX/HTML alongside the per-segment outputs.

In [0]:
state_under_test = "joh_reconciliation"
run_tag_default  = "joh_reconciliation"

dbutils.widgets.text("run_tag", run_tag_default)
run_tag = dbutils.widgets.get("run_tag") or run_tag_default

JOH_SCHEMA     = "ariadm_arm_joh"
JOH_ADJ_TBL    = "hive_metastore.ariadm_arm_joh.raw_adjudicator"
JOH_ROLE_TBL   = "hive_metastore.ariadm_arm_joh.raw_adjudicatorrole"
JOH_GOLD_TBL   = "hive_metastore.ariadm_arm_joh.gold_judicial_officer_with_json"
EXCLUDED_ROLES = [7, 8]
EXPECTED_N     = 1860

In [0]:
import os, sys, time as perf_time
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.functions import col

run_user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
test_results_path = "/Workspace/Users/" + run_user + "/Results/JOH_Reconciliation_Tests"
os.makedirs(test_results_path, exist_ok=True)

for _mod in list(sys.modules.keys()):
    if _mod.startswith("Test_Functions") or _mod == "models.test_result":
        del sys.modules[_mod]

from models.test_result import TestResult
from Test_Functions.report_helpers import (
    build_report_folder,
    write_csv, write_xlsx, write_html,
    count_by_status,
    create_run, create_results, create_perf_results,
)
print(f"User: {run_user}")
print(f"state_under_test: {state_under_test}")

In [0]:
#######################
# Spark Config — wire up the service-principal credentials so we can read ADLS-backed tables.
#######################
config = spark.read.option("multiline", "true").json("dbfs:/configs/config.json")
first_row = config.first()
env = first_row["env"].strip().lower()
lz_key = first_row["lz_key"].strip().lower()
keyvault_name = f"ingest{lz_key}-meta002-{env}"

client_secret = dbutils.secrets.get(scope=keyvault_name, key="SERVICE-PRINCIPLE-CLIENT-SECRET")
tenant_id     = dbutils.secrets.get(scope=keyvault_name, key="SERVICE-PRINCIPLE-TENANT-ID")
client_id     = dbutils.secrets.get(scope=keyvault_name, key="SERVICE-PRINCIPLE-CLIENT-ID")

for storage in (f"ingest{lz_key}curated{env}", f"ingest{lz_key}xcutting{env}",
                f"ingest{lz_key}raw{env}", f"ingest{lz_key}landing{env}", f"ingest{lz_key}external{env}"):
    spark.conf.set(f"fs.azure.account.auth.type.{storage}.dfs.core.windows.net", "OAuth")
    spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
    spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage}.dfs.core.windows.net", client_id)
    spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage}.dfs.core.windows.net", client_secret)
    spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
print(f"env={env}  lz_key={lz_key}")

In [0]:
all_test_results = []
perf_timings     = []
run_start_datetime = datetime.utcnow()

def _try_load(fq_table):
    try:
        return spark.table(fq_table)
    except Exception as e:
        print(f"  ! cannot load {fq_table}: {str(e)[:160]}")
        return None

def _run_timed(test_name, fn, *args, **kwargs):
    before = len(all_test_results)
    t0 = perf_time.perf_counter()
    fn(*args, **kwargs)
    elapsed = perf_time.perf_counter() - t0
    perf_timings.append({
        "test_from_state": state_under_test,
        "test_name": test_name,
        "elapsed_seconds": elapsed,
        "result_count": len(all_test_results) - before,
    })

def _add(test_name, field, status, message="", description=""):
    all_test_results.append(TestResult(
        test_name=test_name, test_field=field, test_from_state=state_under_test,
        status=status, message=message, description=description,
    ))

counts = {}

## Test 1 — Gather counts

Loads `raw_adjudicator`, `raw_adjudicatorrole`, and `gold_judicial_officer_with_json`. Computes:

- **total_adjudicators**: every row in `raw_adjudicator`
- **after_segmentation**: distinct AdjudicatorIds left after applying the LLD filter `Role NOT IN (7, 8) OR Role IS NULL`
- **gold**: row count in the gold table

In [0]:
def test_gather_counts():
    desc = "Gather raw / segmentation / gold counts for JOH"
    adj_df  = _try_load(JOH_ADJ_TBL)
    role_df = _try_load(JOH_ROLE_TBL)
    gold_df = _try_load(JOH_GOLD_TBL)

    if adj_df is None:
        _add("test_gather_counts", "raw_adjudicator", "NO_DATA", f"could not load {JOH_ADJ_TBL}", desc)
        counts["total"] = None
    else:
        counts["total"] = adj_df.select("AdjudicatorId").distinct().count()
        _add("test_gather_counts", "raw_adjudicator", "PASS",
             f"{counts['total']} distinct AdjudicatorIds in raw", desc)

    if adj_df is None or role_df is None:
        _add("test_gather_counts", "after_segmentation", "NO_DATA",
             "raw_adjudicator or raw_adjudicatorrole not loadable", desc)
        counts["after_seg"] = None
    else:
        # LLD: WHERE jr.Role NOT IN (7, 8) OR jr.Role IS NULL
        # In other words: keep an AdjudicatorId UNLESS every one of its roles is in EXCLUDED_ROLES.
        joined = adj_df.alias("a").join(
            role_df.alias("jr"),
            col("a.AdjudicatorId") == col("jr.AdjudicatorId"),
            "left",
        )
        passing = joined.filter(
            (~col("jr.Role").isin(*EXCLUDED_ROLES)) | col("jr.Role").isNull()
        )
        counts["after_seg"] = passing.select("a.AdjudicatorId").distinct().count()
        _add("test_gather_counts", "after_segmentation", "PASS",
             f"{counts['after_seg']} distinct AdjudicatorIds pass LLD filter (Role NOT IN {EXCLUDED_ROLES} OR NULL)",
             desc)

    if gold_df is None:
        _add("test_gather_counts", "gold", "NO_DATA", f"could not load {JOH_GOLD_TBL}", desc)
        counts["gold"] = None
    else:
        counts["gold"] = gold_df.count()
        _add("test_gather_counts", "gold", "PASS",
             f"{counts['gold']} rows in gold_judicial_officer_with_json", desc)

_run_timed("test_gather_counts", test_gather_counts)

## Test 2 — Reconciliation

Segmentation-pass count must equal gold count. A mismatch indicates the gold transformation lost or duplicated judicial officers.

In [0]:
def test_reconciliation():
    desc = "count(Adjudicator passing segmentation) == count(gold_judicial_officer_with_json)"
    if counts.get("after_seg") is None or counts.get("gold") is None:
        _add("test_reconciliation", "EQUATION", "NO_DATA",
             "segmentation or gold count missing", desc)
        return
    diff = counts["after_seg"] - counts["gold"]
    _add("test_reconciliation", "EQUATION",
         "PASS" if diff == 0 else "FAIL",
         f"segmentation={counts['after_seg']} gold={counts['gold']} diff={diff}", desc)

_run_timed("test_reconciliation", test_reconciliation)

## Test 3 — AdjudicatorId uniqueness in gold

Every row in gold should have a distinct AdjudicatorId.

In [0]:
def test_gold_uniqueness():
    desc = "every gold row has a unique AdjudicatorId"
    gold_df = _try_load(JOH_GOLD_TBL)
    if gold_df is None:
        _add("test_gold_uniqueness", "gold", "NO_DATA", "gold not loadable", desc)
        return
    total = gold_df.count()
    distinct = gold_df.select("AdjudicatorId").distinct().count()
    if total == distinct:
        _add("test_gold_uniqueness", "gold", "PASS",
             f"{total} rows, all AdjudicatorIds unique", desc)
    else:
        dups = (gold_df.groupBy("AdjudicatorId").count()
                .filter(col("count") > 1)
                .limit(5).collect())
        ex = [(r.AdjudicatorId, r["count"]) for r in dups]
        _add("test_gold_uniqueness", "gold", "FAIL",
             f"total={total} distinct={distinct} duplicate examples: {ex}", desc)

_run_timed("test_gold_uniqueness", test_gold_uniqueness)

## Test 4 — Expected-volume sanity check (informational)

LLD estimated ~1,860 records. This emits an informational PASS regardless — just a flag if the gold size has diverged significantly from the LLD baseline.

In [0]:
def test_volume_baseline():
    desc = "informational: how does gold size compare to the LLD estimate?"
    if counts.get("gold") is None:
        _add("test_volume_baseline", "vs_LLD_estimate", "NO_DATA", "gold count missing", desc)
        return
    n = counts["gold"]
    pct = (n - EXPECTED_N) / EXPECTED_N * 100 if EXPECTED_N else 0
    _add("test_volume_baseline", "vs_LLD_estimate", "PASS",
         f"gold={n}  LLD-estimate={EXPECTED_N}  diff={n - EXPECTED_N} ({pct:+.1f}%)", desc)

_run_timed("test_volume_baseline", test_volume_baseline)

## Flow summary

Visual breakdown of JOH counts from raw → segmentation → gold. Each leg is also added to the test report as a `flow_summary` row for the CSV / XLSX.

In [0]:
def _fmt_n(n):
    return f"{n:>10,}" if isinstance(n, int) else f"{'?':>10}"

def emit_flow_summary():
    total = counts.get("total")
    seg   = counts.get("after_seg")
    gold  = counts.get("gold")

    desc = "JOH record flow: raw_adjudicator → segmentation (Role filter) → gold_judicial_officer_with_json"
    _add("flow_summary", "01_raw_adjudicator",
         "PASS" if isinstance(total, int) else "NO_DATA",
         f"raw_adjudicator distinct: {_fmt_n(total).strip()}", desc)
    _add("flow_summary", "02_after_segmentation",
         "PASS" if isinstance(seg, int) else "NO_DATA",
         f"after segmentation filter: {_fmt_n(seg).strip()}", desc)
    _add("flow_summary", "03_gold",
         "PASS" if isinstance(gold, int) else "NO_DATA",
         f"gold_judicial_officer_with_json: {_fmt_n(gold).strip()}", desc)

    if isinstance(seg, int) and isinstance(gold, int):
        diff = seg - gold
        _add("flow_summary", "04_reconciliation",
             "PASS" if diff == 0 else "FAIL",
             f"segmentation - gold = {diff}", desc)

    print()
    print("=" * 78)
    print("  JOH FLOW SUMMARY: raw_adjudicator -> segmentation -> gold")
    print("=" * 78)
    print(f"  raw_adjudicator (distinct AdjudicatorId) {'.' * 28} {_fmt_n(total)}")
    print(f"  │")
    print(f"  ├─ Filter:  Role NOT IN {EXCLUDED_ROLES} OR Role IS NULL")
    print(f"  ▼")
    print(f"  after segmentation {'.' * 50} {_fmt_n(seg)}")
    print(f"  │")
    print(f"  ├─ Pipeline transformation")
    print(f"  ▼")
    print(f"  gold_judicial_officer_with_json {'.' * 37} {_fmt_n(gold)}")
    print()
    if isinstance(seg, int) and isinstance(gold, int):
        diff = seg - gold
        tick = "✓" if diff == 0 else "✗"
        print(f"  Reconciliation: segmentation={seg:,}  gold={gold:,}  diff={diff} {tick}")
    print(f"  LLD baseline estimate: ~{EXPECTED_N:,}")
    if isinstance(gold, int):
        d = gold - EXPECTED_N
        print(f"  vs estimate: {d:+,} ({d/EXPECTED_N*100:+.1f}%)")
    print("=" * 78)

_run_timed("flow_summary", emit_flow_summary)

## Save results & headline banner

In [0]:
status_counts = count_by_status(all_test_results)
n_fail = status_counts.get("FAIL", 0); n_err = status_counts.get("ERROR", 0)
n_pass = status_counts.get("PASS", 0); n_nodata = status_counts.get("NO_DATA", 0)
n_total = status_counts.get("TOTAL", 0)
elapsed = sum(p["elapsed_seconds"] for p in perf_timings)

overall = "PASS" if (n_fail == 0 and n_err == 0) else "FAIL"
banner_msg = f"Total={n_total} Pass={n_pass} Fail={n_fail} Error={n_err} NoData={n_nodata} Elapsed={elapsed:.1f}s"

all_test_results.append(TestResult(
    test_name="00_OVERALL_SUMMARY", test_field="00_OVERALL", test_from_state=state_under_test,
    status=overall, message=banner_msg,
    description="Overall verdict for this JOH reconciliation run.",
))

run_id = create_run(spark, run_user, run_start_datetime, "JOH_Reconciliation_Tests",
                    run_tag, overall, state_under_test)
n_results = create_results(spark, run_id, all_test_results)
n_perf    = create_perf_results(spark, run_id, perf_timings, state_under_test)

folder, ts, _safe = build_report_folder(test_results_path, state_under_test)
for fmt_name, fn in (("csv", write_csv), ("xlsx", write_xlsx), ("html", write_html)):
    try:
        if fmt_name == "html":
            fn(all_test_results, state_under_test, folder, ts, counts=status_counts)
        else:
            fn(all_test_results, state_under_test, folder, ts)
    except Exception as _e:
        print(f"  ! {fmt_name} write failed (continuing): {type(_e).__name__}: {str(_e)[:160]}")

print("=" * 70)
print(f"  OVERALL RESULT: {overall}    ({state_under_test})")
print("=" * 70)
print(f"  {banner_msg}")
print(f"  run_id: {run_id}   results written: {n_results}   perf rows: {n_perf}")
print(f"  Folder: {folder}")
print("=" * 70)
for r in all_test_results:
    if r.status in ("FAIL", "ERROR") and r.test_name != "00_OVERALL_SUMMARY":
        print(f"    [{r.test_name}] {r.test_field} -- {r.message}")